# ==============================================================================
# 🧠 CLINICAL NEUROLOGY: STROKE RISK VIA BALANCED RANDOM FOREST CLASSIFIER
# ==============================================================================
### Core Principles:
Stroke prediction involves extreme class imbalance ($pprox 4.8\%$ positive cases).
We utilize Random Forest with `class_weight='balanced'` and `oob_score=True`.
Each tree independently votes to flag cerebrovascular risk.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Neurology Analytics Environment ready.")


✅ STEP 1: Neurology Analytics Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle Healthcare Stroke Dataset (5,110 patients).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/stroke_prediction/healthcare-dataset-stroke-data.csv'
df = pd.read_csv(data_path)

df = df.drop(columns=['id'])
df = df[df['gender'] != 'Other'].copy()

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} patients, {df.shape[1]} attributes")
print(f"Stroke distribution: {df['stroke'].value_counts().to_dict()}")
df.head(3)


📊 Dataset Shape: 5,109 patients, 11 attributes
Stroke distribution: {0: 4860, 1: 249}


,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `stroke`.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df.drop(columns=['stroke'])
y = df['stroke']

print(f"Missing attributes:\n{X.isnull().sum()[X.isnull().sum() > 0]}")
print(f"Stroke Prevalence: {y.mean()*100:.2f}% (Extreme Imbalance!)")


Missing attributes:
bmi    201
dtype: int64
Stroke Prevalence: 4.87% (Extreme Imbalance!)


## 🔒 STEP 6: STRATIFIED SPLIT
80/20 train/test split preserving rare stroke cases.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Patients: {X_train.shape[0]:,} (Strokes: {y_train.sum()})")
print(f"Test Patients : {X_test.shape[0]:,} (Strokes: {y_test.sum()})")


Train Patients: 4,087 (Strokes: 199)
Test Patients : 1,022 (Strokes: 50)


## ⚙️ STEP 7: SCALE-INVARIANT PREPROCESSOR
Imputing missing BMI with median and encoding categorical factors.


In [ ]:
# ==============================================================================
# ⚙️ STEP 7: TREE-OPTIMIZED UNIVERSAL PREPROCESSOR (SCALE-INVARIANT)
# ==============================================================================
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

def build_tree_preprocessor(X):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])
    
    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ])
    
    transformers = []
    if len(num_cols) > 0:
        transformers.append(('num', num_pipeline, num_cols))
    if len(cat_cols) > 0:
        transformers.append(('cat', cat_pipeline, cat_cols))
        
    if len(transformers) == 0:
        return 'passthrough'
        
    return ColumnTransformer(transformers=transformers, remainder='drop')

preprocessor = build_tree_preprocessor(X_train)
print(f"✅ STEP 7: Tree-Optimized Universal Preprocessor Ready.")

✅ STEP 7: Tree-Optimized Universal Preprocessor Ready.


## ⚠️ STEP 8: THE CLASS-IMBALANCE TRAP
Standard trees fail on rare minority targets. Setting `class_weight='balanced'` penalizes false negatives.


In [ ]:
# STEP 8: UNCONSTRAINED BASELINE FIT (OVERFITTING PROOF)
pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(
        n_estimators=150,
        class_weight='balanced',
        oob_score=True,
        random_state=42,
        n_jobs=-1
    ))
])
pipe_rf.fit(X_train, y_train)

print(f"Random Forest Out-of-Bag (OOB) Score: {pipe_rf.named_steps['rf'].oob_score_*100:.2f}%")
print(f"Test Set ROC-AUC                     : {roc_auc_score(y_test, pipe_rf.predict_proba(X_test)[:, 1])*100:.2f}%")


Random Forest Out-of-Bag (OOB) Score: 93.12%
Test Set ROC-AUC                     : 79.22%


/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:589: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


## 🎯 STEP 9: HYPERPARAMETER TUNING VIA 5-FOLD CV
Optimizing `max_depth` and `min_samples_leaf` for medical ROC-AUC.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'rf__n_estimators': [100, 150],
    'rf__max_depth': [6, 10, 14],
    'rf__min_samples_leaf': [2, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])
grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV ROC-AUC: {grid.best_score_*100:.2f}%")


🏆 Best Hyperparameters: {'rf__max_depth': 14, 'rf__min_samples_leaf': 5, 'rf__n_estimators': 150}
🎯 Best 5-Fold CV ROC-AUC: 83.47%


## ⚔️ STEP 10: ALGORITHM DUEL (Random Forest vs Single Tree)
Evaluating ensemble ROC-AUC against a single decision tree.


In [8]:
# STEP 10: BENCHMARK
pipe_single = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])
pipe_single.fit(X_train, y_train)

roc_single = roc_auc_score(y_test, pipe_single.predict_proba(X_test)[:, 1])
roc_rf = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])

print("="*65)
print(f"Single Decision Tree -> ROC-AUC: {roc_single*100:.2f}%")
print(f"Random Forest (Tuned)-> ROC-AUC: {roc_rf*100:.2f}% (+{(roc_rf - roc_single)*100:.2f}% ROC gain!)")
print("="*65)


Single Decision Tree -> ROC-AUC: 58.53%
Random Forest (Tuned)-> ROC-AUC: 81.42% (+22.88% ROC gain!)


## 📊 STEP 11: FEATURE IMPORTANCE & ERROR DIAGNOSTICS & CONFUSION MATRIX
Comprehensive neurology risk report.


In [9]:
# STEP 11: FEATURE IMPORTANCE & ERROR DIAGNOSTICS
y_pred_rf = best_model.predict(X_test)
print("📋 CLINICAL STROKE RISK CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_rf, target_names=['No Stroke', 'Stroke Risk']))

cm = confusion_matrix(y_test, y_pred_rf)
print(f"Confusion Matrix:\n{cm}")


📋 CLINICAL STROKE RISK CLASSIFICATION REPORT:
              precision    recall  f1-score   support

   No Stroke       0.98      0.84      0.90       972
 Stroke Risk       0.18      0.70      0.29        50

    accuracy                           0.83      1022
   macro avg       0.58      0.77      0.60      1022
weighted avg       0.94      0.83      0.87      1022

Confusion Matrix:
[[813 159]
 [ 15  35]]


## 💾 STEP 12 & 13: SERIALIZATION & CLINICAL INFERENCE SMOKE TEST
Deploying the model pipeline and benchmarking sub-2ms bedside stroke screening.


In [10]:
# STEP 12 & 13: PRODUCTION SERIALIZATION & LATENCY SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/stroke_risk_random_forest_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_patient = pd.DataFrame([{
    'gender': 'Male',
    'age': 67.0,
    'hypertension': 1,
    'heart_disease': 1,
    'ever_married': 'Yes',
    'work_type': 'Private',
    'residence_type': 'Urban',
    'avg_glucose_level': 228.69,
    'bmi': 36.6,
    'smoking_status': 'formerly smoked'
}])

t0 = time.perf_counter()
pred_stroke = loaded_pipe.predict(sample_patient)[0]
stroke_prob = loaded_pipe.predict_proba(sample_patient)[0][1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🧠 LIVE BEDSIDE STROKE RISK AUDIT:")
print(f"   Clinical Assessment: {'🚨 HIGH STROKE RISK TIER' if pred_stroke == 1 else '✅ LOW STROKE RISK'}")
print(f"   Risk Probability   : {stroke_prob*100:.2f}%")
print(f"   Inference Latency  : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/stroke_risk_random_forest_pipeline.joblib (846,513 bytes)



🧠 LIVE BEDSIDE STROKE RISK AUDIT:
   Clinical Assessment: 🚨 HIGH STROKE RISK TIER
   Risk Probability   : 84.60%
   Inference Latency  : 59.209 ms (SLA < 2.0ms: CHECK)
